In [ ]:
import MotoricaInterface as mi; assert(mi.__version__=="0.0.8") # last time launched
from MotoricaInterface.crc import get_crc
import numpy as np
import time
import pandas as pd

In [2]:
df_ports = mi.utils.list_devices()
df_ports

,device,name,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,/dev/cu.BLTH,cu.BLTH,n/a,n/a,NaN,NaN,None,None,None,None,None
1,/dev/cu.ESP32-BT-Slave,cu.ESP32-BT-Slave,n/a,n/a,NaN,NaN,None,None,None,None,None
2,/dev/cu.BoseQC35II,cu.BoseQC35II,n/a,n/a,NaN,NaN,None,None,None,None,None
3,/dev/cu.Bluetooth-Incoming-Port,cu.Bluetooth-Incoming-Port,n/a,n/a,NaN,NaN,None,None,None,None,None
4,/dev/cu.usbserial-14410,cu.usbserial-14410,USB Serial,USB VID:PID=1A86:7523 LOCATION=20-4.1,6790.0,29987.0,None,20-4.1,None,USB Serial,None


In [3]:
import ipywidgets as widgets
from IPython.display import display

# Create the dropdown widget
style = {'description_width': 'initial'}
dropdown = widgets.Dropdown(
    options=df_ports.device.unique(),
    description='Select Device:',
    disabled=False,
    style=style
)

# Create a message field
message = widgets.Label(value="")

ser = None
def select(b):
    global ser
    ser = mi.FestSerialInterface(dropdown.value, debug=False, logging_enabled=False)
    message.value = f"Selected: {dropdown.value}\n{ser}"
    

# Create the button widget
button = widgets.Button(
    description='Launch Device',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Launch the device'
)

# Link the button to the select function
button.on_click( select)

# Use HBox to align the dropdown and button horizontally
hbox = widgets.HBox([dropdown, button])

# Use VBox to stack the HBox and the message field vertically
vbox = widgets.VBox([hbox, message])

# Display the VBox containing the HBox and message field
display(vbox)

# Start parsing

In [4]:
ser.start()

Start Serial Reading


# Simple Pack Counter

In [ ]:
ser.DEBUG = False
pack_counter = 0;
t_start = time.time()
t_last_pack = 0;

ser.flush_rx_packs()

while True:
    if (len(ser.rx_pack)):
        now = time.time()
        ts, RX = ser.rx_pack.pop()
        print(f'N={pack_counter}\tFrom thread {ts-ser.thread_start_time:.3f}\tFrom main {time.time() - t_start:.2f}\tPACK_HEADER {RX[:2]}\tTS_DIFF={(now - t_last_pack)*1000:.0f}', 
              end = ' '*20 + '\r')
        t_last_pack = now
        pack_counter+=1
    else:
        time.sleep(0.01) 

KeyboardInterrupt: 

# Simple movements open/closed

In [ ]:
ser.prot_open()
time.sleep(2)
while True:
    for n in range(5):        
        ser.prot_close()
        time.sleep(2)
        ser.prot_open()
        time.sleep(2)

[ 1 19  0 18]
[  1  15   0 179]
[ 1 19  0 18]
[ 1 16  0 71]


KeyboardInterrupt: 

# Absolute finger position

In [5]:
ser.prot_open()
for i in range(6):
    for j in [0, 25, 50, 75, 100, 75, 50, 25, 0]:
        fingers = np.zeros(6, dtype='uint8')
        fingers[i] = j
        ser.prot_abs_state(*fingers)
        time.sleep(0.5)

[ 1 19  0 18]
[  1  15   0 179]
[  1 187   6   0   0   0   0   0   0 171]
[  1 187   6   0   0   0   0  25   0 245]


KeyboardInterrupt: 

# Relative Movements

In [19]:
ser.prot_rel_move(index=100)
time.sleep(1.5)
ser.prot_rel_move(index=-100)
time.sleep(1.5)

ser.prot_rel_move(thumb=100)
time.sleep(1.5)
ser.prot_rel_move(thumb=-100)
time.sleep(1.5)



[1, 170, 6, 254, 127, 127, 127, 127, 127, 74]
[1, 170, 6, 0, 127, 127, 127, 127, 127, 43]
[1, 170, 6, 127, 127, 127, 127, 254, 127, 124]
[1, 170, 6, 127, 127, 127, 127, 0, 127, 57]


# Gesture commands

In [23]:
for g in ser.gesture_names:
    print(g)
    ser.prot_exec_gesture(g)
    time.sleep(1.5)


grab
[  1 187   6 100 100 100 100 100 100 122]
open
[  1 187   6   0   0   0   0   0   0 171]
neutral
[  1 187   6  25  25  25  25  25  25 217]
indication
[  1 187   6   0 100 100 100   0   0   0]
pinch
[  1 187   6  50   0   0   0  50 100 144]
